# Bayesian Statistics with PyMC

Probabilistic programming lets you express complex models as code and automatically infer posterior distributions. PyMC3/PyMC is the gold standard for Bayesian data analysis in Python.

**Topics:** Bayesian linear regression, hierarchical models, MCMC diagnostics, posterior predictive checks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

try:
    import pymc as pm
    import arviz as az
    print(f'PyMC {pm.__version__} available')
    print(f'ArviZ {az.__version__} available')
    PYMC_AVAILABLE = True
except ImportError:
    PYMC_AVAILABLE = False
    print('PyMC not installed. Install: pip install pymc arviz')
    print('Falling back to manual MCMC implementations')

## 1. Bayesian vs Frequentist Mindset

In [ ]:
# Frequentist vs Bayesian: coin flip example
# Observed: 7 heads in 10 flips

n_flips = 10
n_heads = 7

# --- Frequentist approach ---
# H0: p = 0.5, test statistic
result = stats.binomtest(n_heads, n_flips, p=0.5, alternative='two-sided')
print('=== Frequentist Approach ===')
print(f'Observed: {n_heads}/{n_flips} heads')
print(f'p-value: {result.pvalue:.4f}')
print(f'Conclusion: {"Reject H0" if result.pvalue < 0.05 else "Fail to reject H0"} at α=0.05')
print(f'95% CI: [{result.proportion_ci().low:.3f}, {result.proportion_ci().high:.3f}]')
print()

# --- Bayesian approach ---
# Prior: Beta(1,1) — uniform (no prior knowledge)
# Likelihood: Binomial
# Posterior: Beta(1+7, 1+3) = Beta(8, 4)

from scipy.stats import beta as beta_dist

prior_alpha, prior_beta = 1, 1
post_alpha = prior_alpha + n_heads
post_beta  = prior_beta + (n_flips - n_heads)

post_mean = post_alpha / (post_alpha + post_beta)
post_mode = (post_alpha - 1) / (post_alpha + post_beta - 2)
ci_95 = beta_dist.ppf([0.025, 0.975], post_alpha, post_beta)
prob_fair = beta_dist.cdf(0.6, post_alpha, post_beta) - beta_dist.cdf(0.4, post_alpha, post_beta)

print('=== Bayesian Approach ===')
print(f'Prior: Beta({prior_alpha}, {prior_beta}) — uniform')
print(f'Posterior: Beta({post_alpha}, {post_beta})')
print(f'Posterior mean: {post_mean:.3f}')
print(f'Posterior mode: {post_mode:.3f}')
print(f'95% Credible interval: [{ci_95[0]:.3f}, {ci_95[1]:.3f}]')
print(f'P(0.4 < p < 0.6): {prob_fair:.3f} — probability coin is "fair"')

# Visualize prior, likelihood, posterior
theta = np.linspace(0, 1, 500)
prior = beta_dist.pdf(theta, prior_alpha, prior_beta)
likelihood = stats.binom.pmf(n_heads, n_flips, theta)
posterior = beta_dist.pdf(theta, post_alpha, post_beta)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, y, title, color in zip(axes,
    [prior, likelihood/likelihood.max()*5, posterior],
    [f'Prior: Beta({prior_alpha},{prior_beta})', 'Likelihood (scaled)', f'Posterior: Beta({post_alpha},{post_beta})'],
    ['steelblue', 'orange', 'green']):
    ax.fill_between(theta, y, alpha=0.4, color=color)
    ax.plot(theta, y, color=color, lw=2)
    ax.set_title(title); ax.set_xlabel('p (coin bias)')
    ax.axvline(0.5, color='red', ls='--', alpha=0.7, label='Fair (p=0.5)')
    ax.legend(fontsize=8)
plt.suptitle('Bayesian Updating: Prior × Likelihood → Posterior', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Bayesian Linear Regression (From Scratch)

In [ ]:
# Bayesian linear regression via Metropolis-Hastings MCMC
# Model: y = alpha + beta*x + eps,  eps ~ N(0, sigma^2)

# Generate data
n = 50
true_alpha, true_beta, true_sigma = 2.0, 1.5, 1.0
x = np.random.uniform(0, 5, n)
y = true_alpha + true_beta * x + np.random.normal(0, true_sigma, n)

def log_posterior(alpha, beta, log_sigma, x, y):
    """Log posterior (= log prior + log likelihood) up to constant."""
    sigma = np.exp(log_sigma)
    
    # Priors: Normal(0,10) for alpha, beta; HalfNormal(1) for sigma
    log_prior = (stats.norm.logpdf(alpha, 0, 10) +
                 stats.norm.logpdf(beta, 0, 10) +
                 stats.norm.logpdf(sigma, 0, 1))  # HalfNormal approx
    
    # Likelihood
    mu = alpha + beta * x
    log_lik = stats.norm.logpdf(y, mu, sigma).sum()
    
    return log_prior + log_lik

# Metropolis-Hastings sampler
n_samples = 10000
n_burnin  = 2000
step_sizes = np.array([0.1, 0.05, 0.05])  # for alpha, beta, log_sigma

# Initialize chain
current = np.array([0.0, 0.0, 0.0])  # alpha, beta, log_sigma
samples = np.zeros((n_samples, 3))
accepted = 0

for i in range(n_samples):
    # Propose
    proposal = current + np.random.normal(0, step_sizes)
    
    # Acceptance ratio
    log_ratio = log_posterior(*proposal, x, y) - log_posterior(*current, x, y)
    
    if np.log(np.random.random()) < log_ratio:
        current = proposal
        accepted += 1
    
    samples[i] = current

# Discard burn-in
samples_burned = samples[n_burnin:]
acceptance_rate = accepted / n_samples

alpha_samples = samples_burned[:, 0]
beta_samples  = samples_burned[:, 1]
sigma_samples = np.exp(samples_burned[:, 2])

print(f'Acceptance rate: {acceptance_rate:.2%} (target: 20-40%)')
print(f'\nParameter estimates:')
print(f'  alpha: true={true_alpha:.1f}  posterior mean={alpha_samples.mean():.3f} ± {alpha_samples.std():.3f}')
print(f'  beta:  true={true_beta:.1f}  posterior mean={beta_samples.mean():.3f} ± {beta_samples.std():.3f}')
print(f'  sigma: true={true_sigma:.1f}  posterior mean={sigma_samples.mean():.3f} ± {sigma_samples.std():.3f}')

# Trace plots + posterior distributions
fig, axes = plt.subplots(3, 2, figsize=(14, 9))
param_labels = ['α (intercept)', 'β (slope)', 'σ (noise)']
true_vals = [true_alpha, true_beta, true_sigma]
param_samples = [alpha_samples, beta_samples, sigma_samples]

for i, (label, samp, true_val) in enumerate(zip(param_labels, param_samples, true_vals)):
    axes[i, 0].plot(samp[:1000], lw=0.5, alpha=0.7, color='steelblue')
    axes[i, 0].axhline(true_val, color='red', ls='--', label=f'True={true_val}')
    axes[i, 0].set_title(f'Trace: {label}'); axes[i, 0].legend(fontsize=8)
    
    axes[i, 1].hist(samp, bins=60, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    axes[i, 1].axvline(true_val, color='red', ls='--', label=f'True={true_val}')
    axes[i, 1].axvline(samp.mean(), color='orange', ls='-', label=f'Post mean={samp.mean():.3f}')
    ci = np.percentile(samp, [2.5, 97.5])
    axes[i, 1].axvspan(ci[0], ci[1], alpha=0.2, color='orange', label=f'95% CrI')
    axes[i, 1].set_title(f'Posterior: {label}'); axes[i, 1].legend(fontsize=8)

plt.suptitle('Bayesian Linear Regression — MCMC Results', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. PyMC Model Specification

In [ ]:
# Show PyMC syntax even if not installed
print('=== PyMC Bayesian Linear Regression ===')
print('Install: pip install pymc arviz')

pymc_code = '''
import pymc as pm
import arviz as az
import numpy as np

# Standardize inputs for better MCMC mixing
x_std = (x - x.mean()) / x.std()
y_std = (y - y.mean()) / y.std()

with pm.Model() as linear_model:
    # Priors
    alpha = pm.Normal("alpha", mu=0, sigma=1)   # intercept
    beta  = pm.Normal("beta",  mu=0, sigma=1)   # slope
    sigma = pm.HalfNormal("sigma", sigma=1)     # noise
    
    # Likelihood
    mu = alpha + beta * x_std
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_std)
    
    # Inference with NUTS (No-U-Turn Sampler — auto-tuned HMC)
    idata = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        target_accept=0.9,  # adapt step size
        return_inferencedata=True
    )

# Diagnostics
az.plot_trace(idata)          # trace + posterior plots
az.plot_posterior(idata)      # posterior with CrI
az.summary(idata)             # R-hat, ESS, etc.

# Posterior predictive check
with linear_model:
    ppc = pm.sample_posterior_predictive(idata)

az.plot_ppc(az.from_pymc(posterior_predictive=ppc, model=linear_model))
# Checks if model-generated data looks like observed data
'''
print(pymc_code)

print('MCMC Diagnostics checklist:')
print('  R-hat ≈ 1.0  — chains converged (< 1.01 is excellent)')
print('  ESS > 400    — effective sample size (per chain)')
print('  Trace mix    — traces should look like "caterpillars"')
print('  PPC overlap  — posterior predictive overlaps observed data')

## 4. Hierarchical Models — Partial Pooling

In [ ]:
# Hierarchical model: share information across groups
# Classic example: school test scores across districts

# Generate hierarchical data: 8 groups with different sample sizes
np.random.seed(42)
n_groups = 8
true_grand_mean = 75
true_group_sd = 8
obs_sd = 10

group_sizes = [5, 30, 5, 25, 8, 5, 25, 8]
true_group_means = np.random.normal(true_grand_mean, true_group_sd, n_groups)

observations = []
group_ids = []
for g, (n_g, mu_g) in enumerate(zip(group_sizes, true_group_means)):
    obs = np.random.normal(mu_g, obs_sd, n_g)
    observations.extend(obs)
    group_ids.extend([g] * n_g)

y_obs = np.array(observations)
groups = np.array(group_ids)

# Three approaches: no pooling, complete pooling, partial pooling
# No pooling: estimate each group independently
no_pool_means = [y_obs[groups == g].mean() for g in range(n_groups)]
no_pool_stds  = [y_obs[groups == g].std() / np.sqrt(group_sizes[g]) for g in range(n_groups)]

# Complete pooling: ignore group structure
complete_pool = y_obs.mean()

# Partial pooling (manual Bayesian estimation via empirical Bayes)
grand_mean    = y_obs.mean()
grand_var     = y_obs.var()
group_variances = [y_obs[groups==g].var() for g in range(n_groups)]

# Shrinkage towards grand mean (partial pooling)
lambda_shrink = [obs_sd**2 / (obs_sd**2 + true_group_sd**2 * group_sizes[g])
                 for g in range(n_groups)]
partial_pool_means = [lam * grand_mean + (1-lam) * grp_mean
                      for lam, grp_mean in zip(lambda_shrink, no_pool_means)]

print('Hierarchical Model: Partial Pooling')
print(f'Grand mean: {grand_mean:.1f}')
print()
print(f'{"Group":>5} {"n":>5} {"True":>8} {"No Pool":>9} {"Part Pool":>10} {"Shrink%":>9}')
print('-' * 55)
for g in range(n_groups):
    shrink_pct = lambda_shrink[g] * 100
    print(f'{g:>5} {group_sizes[g]:>5} {true_group_means[g]:>8.1f} '
          f'{no_pool_means[g]:>9.1f} {partial_pool_means[g]:>10.1f} {shrink_pct:>8.1f}%')

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(n_groups)

ax.scatter(x_pos, true_group_means, marker='*', s=200, color='gold', 
           zorder=5, label='True group means', edgecolors='orange')
ax.scatter(x_pos - 0.15, no_pool_means, marker='o', s=80, color='red',
           zorder=4, label='No pooling (MLE per group)')
ax.scatter(x_pos + 0.15, partial_pool_means, marker='D', s=80, color='steelblue',
           zorder=4, label='Partial pooling (shrinkage)')
ax.axhline(complete_pool, color='gray', ls='--', lw=1.5, label=f'Complete pooling ({complete_pool:.1f})')

# Annotate sample sizes
for g, n_g in enumerate(group_sizes):
    ax.annotate(f'n={n_g}', (g, min(no_pool_means[g], partial_pool_means[g]) - 3),
                ha='center', fontsize=9, color='gray')

ax.set_xticks(x_pos); ax.set_xticklabels([f'Group {g}' for g in range(n_groups)])
ax.set_ylabel('Test Score'); ax.set_title('Hierarchical Shrinkage: Partial Pooling vs No Pooling')
ax.legend(); plt.tight_layout(); plt.show()

print()
print('Key insight: Groups with small n shrink heavily toward grand mean.')
print('Groups with large n are mostly influenced by their own data.')
print('Partial pooling reduces MSE vs both extremes — statistical wisdom!')

## 5. Posterior Predictive Checks

In [ ]:
# Posterior predictive checks (PPC):
# Simulate new data from posterior and compare to observed
# If model is wrong, simulated data will look different from observed

# Generate observed data from a heavy-tailed distribution
n_obs = 200
y_data = np.random.standard_t(df=5, size=n_obs) * 3 + 10  # t-distributed with fat tails

# Model 1: Normal likelihood (misspecified)
mu_ml  = y_data.mean()
sig_ml = y_data.std()

# Model 2: Robust (Student-t likelihood) — using MCMC posterior samples
# We'll demonstrate with samples from posterior

# Posterior for Normal model (analytical via conjugate)
n_posterior_samples = 1000

# Normal model PPC
normal_ppc = []
for _ in range(n_posterior_samples):
    mu_draw  = np.random.normal(mu_ml, sig_ml / np.sqrt(n_obs))
    sig_draw = np.abs(np.random.normal(sig_ml, sig_ml / np.sqrt(2*n_obs)))
    normal_ppc.append(np.random.normal(mu_draw, sig_draw, n_obs))

normal_ppc = np.array(normal_ppc)

# Student-t model PPC (approximate)
df_est = 5  # estimated degrees of freedom
t_ppc = []
for _ in range(n_posterior_samples):
    mu_draw  = np.random.normal(mu_ml, sig_ml / np.sqrt(n_obs))
    t_ppc.append(stats.t.rvs(df=df_est, loc=mu_draw, scale=sig_ml * 0.9, size=n_obs))
t_ppc = np.array(t_ppc)

# Visualize PPCs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ppc_samples, model_name, color in zip(
    axes,
    [normal_ppc, t_ppc],
    ['Normal Likelihood (misspecified)', 'Student-t Likelihood (robust)'],
    ['orange', 'steelblue']
):
    # Plot PPCs
    for i in range(100):
        ax.hist(ppc_samples[i], bins=40, density=True, alpha=0.02, color=color)
    
    # Plot observed
    ax.hist(y_data, bins=40, density=True, alpha=0.7, color='black', 
            edgecolor='white', label='Observed data')
    
    ax.set_title(f'PPC: {model_name}')
    ax.set_xlabel('Value')
    ax.legend()

plt.suptitle('Posterior Predictive Check — Model Criticism', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Test statistic for PPC
def ppc_test_statistic(ppc_samples, y_data, statistic_fn):
    """Compare test statistic of observed vs replicated data."""
    t_obs = statistic_fn(y_data)
    t_rep = np.array([statistic_fn(s) for s in ppc_samples])
    p_val = np.mean(t_rep >= t_obs)
    return t_obs, t_rep, p_val

# Check: does the model capture tail behavior?
def kurtosis(x): return stats.kurtosis(x)
def max_val(x):  return np.max(np.abs(x - x.mean()))

print('=== PPC Test Statistics ===')
for statistic, name in [(kurtosis, 'Kurtosis'), (max_val, 'Max abs deviation')]:
    t_obs_n, t_rep_n, p_n = ppc_test_statistic(normal_ppc, y_data, statistic)
    t_obs_t, t_rep_t, p_t = ppc_test_statistic(t_ppc, y_data, statistic)
    print(f'  {name}:')
    print(f'    Observed: {t_obs_n:.3f}')
    print(f'    Normal model p-value: {p_n:.3f}  {"(BAD fit!)" if p_n < 0.05 or p_n > 0.95 else "(OK)"}')
    print(f'    t model   p-value: {p_t:.3f}  {"(BAD fit!)" if p_t < 0.05 or p_t > 0.95 else "(OK)"}')